<a href="https://colab.research.google.com/github/saraakarimii/Datachallenge/blob/main/fitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================== FAST MCMC FIT ON KILONOVA SPECTRUM (mask 7000–10000 Å, smoothed model) ==================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp, trapezoid, simpson
from numpy.polynomial.legendre import leggauss
from scipy.constants import h, c, k  # SI (m, s, J)
from scipy.stats import gaussian_kde

# -----------------------------------------------------------------------------
# 0) Data + direct mask (7000–10000 Å)
# -----------------------------------------------------------------------------
file_path = "AT2017gfo_ENGRAVE_v1.0_XSHOOTER_MJD-57983.969_Phase+1.43d.dat"
df = pd.read_csv(file_path, comment='#', delimiter='\t', header=None)
df.columns = ['wavelength_A', 'flux_tell_corrected', 'flux_not_tell_corrected', 'error']
df = df[df['flux_tell_corrected'] > 0].copy()

fit_lo, fit_hi = 3500.0, 22500.0
m_base = (df['wavelength_A'] >= fit_lo) & (df['wavelength_A'] <= fit_hi)

lam_all  = df.loc[m_base, 'wavelength_A'].values.astype(float)
flux_all = df.loc[m_base, 'flux_tell_corrected'].values.astype(float)
if 'error' in df.columns and np.isfinite(df.loc[m_base, 'error']).any():
    err_all = df.loc[m_base, 'error'].values.astype(float)
else:
    err_all = 0.1*np.median(flux_all)*np.ones_like(flux_all)

# ماسک مستقیم بازه‌ی 7000–10000 Å
bad_region = (lam_all >= 7000.0) & (lam_all <= 10000.0)
m_fit = ~bad_region
print(f"Masked region: 7000–10000 Å  (removed {bad_region.sum()} points)")

lam_A_data = lam_all[m_fit]
flux_data  = flux_all[m_fit]
sigma_data = err_all[m_fit]

# کف خطا (کمی بزرگ‌تر برای لایکلیهود نرم‌تر)
sigma_floor = 0.10*np.median(flux_data)
sigma_data  = np.clip(sigma_data, sigma_floor, None)

# -----------------------------------------------------------------------------
# 1) Constants & observation
# -----------------------------------------------------------------------------
day   = 86400.0
M_sun = 1.989e33            # g
c_cm  = 2.99792458e10       # cm/s
t_obs_days = 1.43

# Fixed distance and heating fraction
D_cm   = 40.0 * 3.086e24      # 40 Mpc (cm)
f_fixed = 1e-6                # fixed

# -----------------------------------------------------------------------------
# 2) Dynamical model (non-relativistic bolometric evolution)
# -----------------------------------------------------------------------------
def non_rel_kilonova(t_days_eval, kappa, Mej_msun, f, V_ej_c, alpha=0.3):
    t_eval = np.asarray(t_days_eval) * day
    Mej = Mej_msun * M_sun
    V_ej = V_ej_c * c_cm
    t_c = np.sqrt(3.0 * kappa * Mej / (4.0 * np.pi * V_ej**2))
    t_min = 1e-6 * day
    t_max = float(np.max(t_eval)) + 1.0

    def dUdt(t, U):
        heating_rate     = (f * c_cm**2 / t_c) * (t_c / t)**(1 + alpha)
        cooling_rate_rad = (np.pi * V_ej * c_cm * t * U[0]) / (kappa * Mej)
        cooling_rate_ad  = 4.0 * U[0] / t
        return [3*Mej*heating_rate/(4*np.pi*V_ej**3 * t**3) - cooling_rate_rad - cooling_rate_ad]

    sol = solve_ivp(
        fun=dUdt, t_span=[t_min, t_max], y0=[0.0],
        t_eval=np.unique(np.append(t_eval, np.linspace(t_min, t_max, 500))),
        method="BDF", rtol=1e-6, atol=1e-10
    )

    U_interp = np.interp(t_eval, sol.t, sol.y[0])
    L_bolo_erg_s = (4.0 * np.pi**2 * V_ej**4 * c_cm * U_interp * t_eval**4) / (3.0 * kappa * Mej)
    R_cm = V_ej * t_eval
    sigma_cgs = 5.670374e-5
    T_eff = (L_bolo_erg_s / (4.0 * np.pi * R_cm**2 * sigma_cgs))**0.25

    return {"time_days": t_days_eval, "t_sec": t_eval,
            "L_bolo_erg_s": L_bolo_erg_s, "R_cm": R_cm, "T_eff": T_eff}

# -----------------------------------------------------------------------------
# 3) Vectorized Planck (W·sr^-1·m^-3)
# -----------------------------------------------------------------------------
def _planck_vec_wm(wav_m, T):
    with np.errstate(over='ignore', invalid='ignore'):
        arg   = h * c / (wav_m * k * T)
        arg   = np.clip(arg, 0, 700)
        denom = np.expm1(arg)
        I     = (2 * h * c**2) / (np.power(wav_m, 5) * denom)
    return np.nan_to_num(I)

# -----------------------------------------------------------------------------
# 4) Relativistic spectrum shape via Gauss–Legendre (vectorized) — Nmu=48
# -----------------------------------------------------------------------------
def _relativistic_shape_GL(wavelengths_nm, v_bb_c, kn_model, t_obs_days, Nmu=48):
    beta = float(v_bb_c)
    lam_nm = np.asarray(wavelengths_nm, dtype=float)
    if beta >= 1.0:
        return np.zeros_like(lam_nm)
    t_obs_sec = t_obs_days * day

    x, w = leggauss(Nmu)
    a, b = beta, 1.0
    mu  = 0.5*(b-a)*x + 0.5*(b+a)
    wmu = 0.5*(b-a)*w

    gamma = 1.0 / np.sqrt(1.0 - beta**2)
    denom   = (1.0 - beta * mu)
    t_emit  = t_obs_sec / denom
    T_emit  = np.interp(t_emit, kn_model['t_sec'], kn_model['T_eff'])
    doppler = (1.0/gamma) / denom
    T_obs   = T_emit * doppler
    geo     = np.power((1.0 - beta)/denom, 2.0)

    lam_m = (lam_nm * 1e-9)[None, :]
    Tgrid = T_obs[:, None]
    geogr = geo[:, None]
    mugr  = mu[:, None]
    wgr   = wmu[:, None]

    I = _planck_vec_wm(lam_m, Tgrid)
    integrand = I * geogr * mugr
    shape = np.sum(integrand * wgr, axis=0)
    return shape

# -----------------------------------------------------------------------------
# 5) L_lambda with normalization  — returns erg s^-1 nm^-1  (Simpson)
# -----------------------------------------------------------------------------
def calculate_L_lambda(wavelengths_nm, v_bb_c, kn_model, t_obs_days, use_simpson=True):
    rel_shape = _relativistic_shape_GL(wavelengths_nm, v_bb_c, kn_model, t_obs_days, Nmu=48)
    t_obs_sec = t_obs_days * day
    L_bolo_erg_s = np.interp(t_obs_sec, kn_model['t_sec'], kn_model['L_bolo_erg_s'])
    L_bolo_W     = L_bolo_erg_s * 1e-7

    lam_m = np.asarray(wavelengths_nm, dtype=float) * 1e-9
    order = np.argsort(lam_m)
    lam_sorted   = lam_m[order]
    shape_sorted = rel_shape[order]

    integ = simpson(shape_sorted, x=lam_sorted) if use_simpson else trapezoid(shape_sorted, x=lam_sorted)
    if integ <= 0 or not np.isfinite(integ):
        return np.zeros_like(rel_shape)

    norm = L_bolo_W / integ
    L_lambda_SI = rel_shape * norm   # W/m
    L_lambda_cgs_per_nm = L_lambda_SI * 1e-2  # 1 W/m = 1e-2 erg s^-1 nm^-1

    inv = np.empty_like(order)
    inv[order] = np.arange(order.size)
    return L_lambda_cgs_per_nm[inv]

# -----------------------------------------------------------------------------
# 6) Flux converter  — L_λ → F_λ
# -----------------------------------------------------------------------------
def calculate_F_lambda(L_lambda_cgs_per_nm, D_cm, per_A=False):
    L = np.asarray(L_lambda_cgs_per_nm, dtype=float)
    D = np.asarray(D_cm, dtype=float)
    if np.any(D <= 0) or not np.all(np.isfinite(D)):
        raise ValueError("D_cm must be positive & finite.")
    F_nm = L / (4.0 * np.pi * D**2)  # erg s^-1 cm^-2 nm^-1
    return F_nm / 10.0 if per_A else F_nm

# -----------------------------------------------------------------------------
# 7) Model wrapper with memoization (no coarse rounding in key)
# -----------------------------------------------------------------------------
_model_cache = {}
def model_flux_at_data_wavelengths(params):
    key = (float(params['kappa']),
           float(params['Mej_msun']),
           float(params['V_ej_c']),
           float(params['alpha']),
           float(t_obs_days), float(f_fixed), float(D_cm))
    if key not in _model_cache:
        # dynamics
        t_model_eval = np.linspace(0.05, max(5.0, t_obs_days*1.2), 300)
        kn = non_rel_kilonova(t_model_eval,
                              kappa=params['kappa'],
                              Mej_msun=params['Mej_msun'],
                              f=f_fixed,
                              V_ej_c=params['V_ej_c'],
                              alpha=params['alpha'])
        # finer wavelength grid (1200 pts) + Simpson
        lamA_grid  = np.linspace(lam_A_data.min(), lam_A_data.max(), 1200)
        lamnm_grid = lamA_grid / 10.0
        L_lambda_cgs_per_nm = calculate_L_lambda(lamnm_grid, params['V_ej_c'], kn, t_obs_days, use_simpson=True)
        F_lambda_nm  = calculate_F_lambda(L_lambda_cgs_per_nm, D_cm, per_A=False)
        F_lambda_A_grid = F_lambda_nm / 10.0
        F_model_A = np.interp(lam_A_data, lamA_grid, F_lambda_A_grid)
        _model_cache[key] = F_model_A
    return _model_cache[key]

# 7b) Wrapper: fit_func (هماهنگ با الگوهای قبلی)
def fit_func(params):
    return model_flux_at_data_wavelengths(params)

# -----------------------------------------------------------------------------
# 8) Priors, likelihood, posterior
# -----------------------------------------------------------------------------
bounds = {
    'kappa':   (0.003,  0.5),    # cm^2/g
    'Mej_msun':(1e-4,   0.2),    # Msun
    'V_ej_c':  (0.03,   0.6),    # v/c
    'alpha':   (0.0,    0.6),
}

def in_bounds(p):
    for k,(lo,hi) in bounds.items():
        v = p[k]
        if not (lo < v < hi):
            return False
    return True

def log_prior(p):
    return 0.0 if in_bounds(p) else -np.inf

def log_likelihood(p):
    model = fit_func(p)
    chi2 = np.sum(((flux_data - model)/sigma_data)**2)
    return -0.5*chi2

def log_probability(p):
    lp = log_prior(p)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(p)

# -----------------------------------------------------------------------------
# 9) Simple Metropolis–Hastings MCMC
# -----------------------------------------------------------------------------
rng = np.random.default_rng(20250929)

def propose(p, step):
    q = p.copy()
    keys = ['kappa','Mej_msun','V_ej_c','alpha']
    for i,k_ in enumerate(keys):
        lo,hi = bounds[k_]
        q[k_] = rng.normal(p[k_], step[i])
        # reflect into bounds
        while (q[k_] <= lo) or (q[k_] >= hi):
            if q[k_] <= lo: q[k_] = 2*lo - q[k_]
            if q[k_] >= hi: q[k_] = 2*hi - q[k_]
    return q

def metropolis_hastings(p0, n_samples=2500, step=(0.003, 0.003, 0.02, 0.02), burnin=400):
    p = p0.copy()
    lp = log_probability(p)
    chain, accepts = [], 0
    for i in range(n_samples + burnin):
        q  = propose(p, step)
        lq = log_probability(q)
        if (lq > lp) or (rng.uniform() < np.exp(lq - lp)):
            p, lp = q, lq
            if i >= burnin: accepts += 1
        if i >= burnin:
            chain.append([p['kappa'], p['Mej_msun'], p['V_ej_c'], p['alpha']])
    chain = np.array(chain)
    acc_rate = accepts / max(1, n_samples)
    return chain, acc_rate

# -----------------------------------------------------------------------------
# 10) Run MCMC
# -----------------------------------------------------------------------------
p0   = dict(kappa=0.015, Mej_msun=0.027, V_ej_c=0.30, alpha=0.10)
step = (0.003, 0.003, 0.02, 0.02)   # tune to ~0.2–0.5 acceptance

chain, acc = metropolis_hastings(p0, n_samples=2500, step=step, burnin=400)
print("Acceptance rate =", acc)
p_best = dict(zip(['kappa','Mej_msun','V_ej_c','alpha'], np.median(chain, axis=0)))
print("Best-fit (median):", p_best)

# یکتایی نسبی نمونه‌ها (برای دیباگ سکوی لایکلیهود)
print("unique kappa (8dp):", len(np.unique(np.round(chain[:,0], 8))))
print("unique Mej   (8dp):", len(np.unique(np.round(chain[:,1], 8))))
print("unique Vej/c (8dp):", len(np.unique(np.round(chain[:,2], 8))))
print("unique alpha (8dp):", len(np.unique(np.round(chain[:,3], 8))))

# -----------------------------------------------------------------------------
# 11) Plots: masked region + best-fit + trace
# -----------------------------------------------------------------------------
plt.figure(figsize=(10,4))
plt.plot(lam_all, flux_all, ".", color='0.8', ms=2, label='All data in window')
plt.axvspan(7000, 10000, color='orange', alpha=0.25, label='Masked: 7000–10000 Å')
plt.plot(lam_A_data, flux_data, 'k.', ms=2.5, label='Used for fit')
plt.yscale('log'); plt.xlabel('Wavelength (Å)'); plt.ylabel('Flux')
plt.title('Direct masking of 7000–10000 Å'); plt.legend(fontsize=9); plt.tight_layout(); plt.show()

F_model_best = fit_func(p_best)
plt.figure(figsize=(10,7))
plt.plot(lam_all, flux_all, ".", color='0.85', ms=2, label='All data')
plt.plot(lam_A_data, flux_data, 'k.', ms=3, label='Used for fit')
plt.plot(lam_A_data, F_model_best, 'r-', lw=2, label='Best-fit Kilonova Model')
plt.yscale('log')
plt.xlabel('Wavelength (Å)')
plt.ylabel(r'Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)')
plt.title('Fit at t=+1.43 d, $D_L$=40 Mpc, $f=10^{-6}$ (fixed), with 7000–10000 Å masked')
plt.legend(); plt.grid(alpha=0.4, ls=':'); plt.tight_layout(); plt.show()

fig,axs = plt.subplots(4,1,figsize=(10,8), sharex=True)
labels = [r'$\kappa$', r'$M_{\rm ej}\,[M_\odot]$', r'$V_{\rm ej}/c$', r'$\alpha$']
for i in range(4):
    axs[i].plot(chain[:,i], lw=0.8)
    axs[i].set_ylabel(labels[i])
axs[-1].set_xlabel('MCMC step')
plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# 12) Goodness-of-fit & residuals (χ²)
# -----------------------------------------------------------------------------
def chi_squared(observed, expected, sigma):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.sum(((observed - expected) / sigma) ** 2)

sigma_used = sigma_data
chi2_value = chi_squared(flux_data, F_model_best, sigma_used)
dof = len(flux_data) - 4   # four free params
red_chi2 = chi2_value / max(1, dof)
print(f"Chi-squared: {chi2_value:.3f}   Reduced χ²: {red_chi2:.3f}  (dof={dof})")

residuals = flux_data - F_model_best
fig, axs = plt.subplots(2, 1, figsize=(10, 9), gridspec_kw={'height_ratios': [2, 1]})
axs[0].plot(lam_all, flux_all, ".", color='0.85', ms=2, label='All data')
axs[0].plot(lam_A_data, flux_data, 'k.', ms=3, label='Used for fit')
axs[0].plot(lam_A_data, F_model_best, '-', color='darkred', lw=2, label='Best-fit model')
axs[0].axvspan(7000, 10000, color='orange', alpha=0.25, label='Masked region')
axs[0].set_yscale('log'); axs[0].set_xlabel('Wavelength (Å)')
axs[0].set_ylabel(r'Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)')
axs[0].legend(fontsize=9); axs[0].grid(alpha=0.3, ls=':')
axs[0].set_title('Best fit + residuals')

axs[1].errorbar(lam_A_data, residuals, yerr=sigma_used, fmt='o', color='k', alpha=0.6, ms=3)
axs[1].axhline(0.0, color='darkred', ls='--', lw=1.2)
axs[1].axvspan(7000, 10000, color='orange', alpha=0.25)
axs[1].set_xlabel('Wavelength (Å)'); axs[1].set_ylabel('Residuals')
axs[1].grid(alpha=0.3, ls=':')
plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# 13) Posterior & pairwise contours (KDE=Scott, ملایم)
# -----------------------------------------------------------------------------
kappa_samples = chain[:,0]
Mej_samples   = chain[:,1]
Vejc_samples  = chain[:,2]
alpha_samples = chain[:,3]

def _nice_range(samples, pad=0.08, qlo=1.0, qhi=99.0, hard=None):
    lo, hi = np.percentile(samples, [qlo, qhi])
    span = hi - lo if hi > lo else (abs(hi) + 1.0)
    lo -= pad * span; hi += pad * span
    if hard is not None:
        lo = max(lo, hard[0]); hi = min(hi, hard[1])
    return float(lo), float(hi)

def plot_posterior_1d(samples, name, ax, units=None, bins=80, color='mediumpurple', prange=None):
    kde = gaussian_kde(samples, bw_method='scott')  # ملایم
    if prange is None: prange = _nice_range(samples)
    x = np.linspace(prange[0], prange[1], 1000)
    y = kde(x)
    ax.hist(samples, bins=bins, range=prange, density=True, alpha=0.45, color=color)
    ax.plot(x, y, color=color, lw=2)
    mean, med, std = np.mean(samples), np.median(samples), np.std(samples)
    if units:
        ann = f'{name} = {mean:.3g} ± {std:.3g} {units}'; xlabel = f'{name} ({units})'
    else:
        ann = f'{name} = {mean:.3g} ± {std:.3g}'; xlabel = f'{name}'
    bbox_props = dict(boxstyle="round,pad=0.3", edgecolor='palegreen', facecolor='white')
    ax.annotate(ann, xy=(0.5, 1.05), xycoords='axes fraction', ha='center', fontsize=10, bbox=bbox_props)
    ax.axvline(med, color='deeppink', ls='--', lw=2)
    ax.set_xlabel(xlabel); ax.set_ylabel('PDF'); ax.set_xlim(prange)

def plot_2d_contour(xs, ys, xlabel, ylabel, ax, x_range=None, y_range=None):
    kde = gaussian_kde([xs, ys], bw_method='scott')
    if x_range is None: x_range = _nice_range(xs)
    if y_range is None: y_range = _nice_range(ys)
    xg, yg = np.mgrid[x_range[0]:x_range[1]:180j, y_range[0]:y_range[1]:180j]
    pos = np.vstack([xg.ravel(), yg.ravel()])
    z = kde(pos).reshape(xg.shape)
    ax.contourf(xg, yg, z, levels=18, cmap='Blues')
    ax.contour(xg, yg, z, levels=6, colors='k', linewidths=0.6, alpha=0.6)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_xlim(x_range); ax.set_ylim(y_range)

kappa_range = _nice_range(kappa_samples, hard=(bounds['kappa'][0], bounds['kappa'][1]))
Mej_range   = _nice_range(Mej_samples,   hard=(bounds['Mej_msun'][0], bounds['Mej_msun'][1]))
Vejc_range  = _nice_range(Vejc_samples,  hard=(bounds['V_ej_c'][0],   bounds['V_ej_c'][1]))
alpha_range = _nice_range(alpha_samples, hard=(bounds['alpha'][0],    bounds['alpha'][1]))

fig, axes = plt.subplots(3, 3, figsize=(12, 12), gridspec_kw={'wspace': 0.3, 'hspace': 0.35})
plt.rcParams.update({"font.family":"serif", "font.size":14, "mathtext.fontset":"stix"})
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]

plot_posterior_1d(kappa_samples, r'$\kappa$',     axes[0,0], units=r'cm$^2$ g$^{-1}$', prange=kappa_range)
plot_posterior_1d(Mej_samples,   r'$M_{\rm ej}$', axes[1,1], units=r'$M_\odot$',       prange=Mej_range)
plot_posterior_1d(alpha_samples, r'$\alpha$',     axes[2,2], prange=alpha_range)
plot_2d_contour(kappa_samples, Mej_samples,   r'$\kappa$ (cm$^2$ g$^{-1}$)', r'$M_{\rm ej}$ ($M_\odot$)', axes[1,0],
                x_range=kappa_range, y_range=Mej_range)
plot_2d_contour(kappa_samples, Vejc_samples,  r'$\kappa$ (cm$^2$ g$^{-1}$)', r'$V_{\rm ej}/c$',            axes[2,0],
                x_range=kappa_range, y_range=Vejc_range)
plot_2d_contour(Mej_samples,   alpha_samples, r'$M_{\rm ej}$ ($M_\odot$)',   r'$\alpha$',                 axes[2,1],
                x_range=Mej_range,   y_range=alpha_range)
# 1D برای Vej/c در خانه‌ی بالا-راست
plot_posterior_1d(Vejc_samples, r'$V_{\rm ej}/c$', axes[0,2], prange=Vejc_range)

axes[0,1].axis('off'); axes[1,2].axis('off')
fig.suptitle('AT2017gfo (+1.43 d) — Kilonova Posterior', fontsize=14, y=0.92)
plt.savefig('AT2017gfo_kilonova_posterior.pdf', dpi=200, bbox_inches='tight')
plt.show()
# ==============================================================================

